# VideoTool cloud GPU whisper — Colab runner (BACKUP)

Same whisper-only core as the Kaggle runner, but I/O is via native `drive.mount`, so there
is no copy step — whisper reads the voice in place and writes the 2 output files back to the
same Drive folder. Use when the Kaggle weekly GPU quota is spent.

Enable a **GPU** runtime (Runtime → Change runtime type → T4 GPU). See
`docs/cloud-gpu-whisper-setup.md`.

**Drive caches (one-time, reused every later session):** the `large-v3` model (~3GB) is
downloaded once into `gdrive:_VIDEOTOOL_SHARED/models/large-v3` and the faster-whisper pip
wheels into `gdrive:_VIDEOTOOL_SHARED/wheelhouse/`. First run pays the download; every later
run loads the model straight from Drive and installs from the cached wheels — no re-download.

## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Fetch cloud core + install videotool


In [ ]:
import shutil
SHARED = '/content/drive/MyDrive/_VIDEOTOOL_SHARED'
WHEELHOUSE = f'{SHARED}/wheelhouse'
shutil.copy(f'{SHARED}/videotool_cloud.py', '.')
import videotool_cloud as vc
vc.setup(wheelhouse=WHEELHOUSE)  # first run installs + caches wheels; later runs reuse them
import torch; assert torch.cuda.is_available(), 'No GPU — pick a T4 runtime.'

## 3. Run whisper on GPU (`large-v3`, float16)

Set `JOB_DIR` to the mounted Drive folder. For a shared drive use
`/content/drive/Shareddrives/...` instead of `MyDrive`.


In [ ]:
JOB_DIR = '/content/drive/MyDrive/1. YOUTUBE AUDIO/.../CHAP N'  # <-- EDIT THIS
MODEL_CACHE = '/content/drive/MyDrive/_VIDEOTOOL_SHARED/models'
caps, chaps = vc.run_whisper(
    JOB_DIR, model='large-v3', device='cuda', compute_type='float16',
    model_cache_dir=MODEL_CACHE,
)
print('captions:', caps)
print('chapters:', chaps)

## 4. Confirm outputs


In [ ]:
import os
print(os.listdir(os.path.join(JOB_DIR,'outputs')))
print(open(caps).read()[:500])

## Full cloud render (LLM job.yaml + NVENC)

Full `/make-video` pipeline on this GPU session — LLM authors `job.yaml`, NVENC renders with
Drive checkpoints, results land in the source folder's `Output/`. Requires **rclone** (bulk I/O;
the FUSE mount is ~60x slower) and an **LLM key** + **rclone remote** in Colab Secrets.
See `docs/cloud-render-setup.md`. Distinct from the whisper cells above.

In [ ]:
# rclone + module fetch. rclone.conf is staged once on the shared Drive (see docs).
!command -v rclone >/dev/null || (curl -s https://rclone.org/install.sh | sudo bash)
import shutil, os
SHARED = '/content/drive/MyDrive/_VIDEOTOOL_SHARED'
for mod in ('videotool_cloud.py', 'cloud_director.py', 'cloud_render_runner.py'):
    shutil.copy(f'{SHARED}/{mod}', '.')
os.makedirs(os.path.expanduser('~/.config/rclone'), exist_ok=True)
shutil.copy(f'{SHARED}/rclone.conf', os.path.expanduser('~/.config/rclone/rclone.conf'))
# Stage the sfx + overlay libraries; Claude names the files, the runner copies the chosen ones in.
for lib in ('sfx', 'overlays'):
    dst = os.path.expanduser(f'~/.local/share/videotool/{lib}')
    os.makedirs(dst, exist_ok=True)
    !rclone copy "gdrive:_VIDEOTOOL_SHARED/{lib}" "$dst" --fast-list
import cloud_render_runner as rr

### Probe: which LLM is best for this pipeline (run with your Secrets loaded)

Calls each provider whose key is in Secrets against a real Vietnamese homograph SFX task and
prints a recommendation. Uses YOUR keys/credit — this is the empirical check of what actually works.

In [ ]:
import cloud_director as cd
# Loads keys from Kaggle/Colab Secrets by name (ANTHROPIC_API_KEY / GEMINI_API_KEY / GLM_API_KEY).
cd.probe_providers()

In [ ]:
SOURCE     = 'gdrive:1. YOUTUBE AUDIO/.../CHAP N'
OUTPUT     = 'gdrive:1. YOUTUBE AUDIO/.../CHAP N/Output'
CHECKPOINT = 'gdrive:_VIDEOTOOL_SHARED/checkpoints/CHAP-N'
CREATIVE   = 'gdrive:_VIDEOTOOL_SHARED/creative/CHAP-N.yaml'   # Claude Code CLI authors this file
REPO_REF   = 'git+https://github.com/pnd4189/video-tool@3f6bd1bd8ded45b5bd5609e4a4de386b9bf4b694'

# Claude authored the creative.yaml -> NO LLM runs here; the box just renders (NVENC).
rr.render_job(SOURCE, OUTPUT, CHECKPOINT, creative_remote=CREATIVE, repo_ref=REPO_REF, local_job='/content/job')
# Rerun after a disconnect -> resumes from the verified checkpoint (no re-author, no re-encode).
# No-Claude fallback: drop creative_remote, add autonomous=True (needs benchmarks init in setup).